MILESTONE 02: KPI ENGINEERING & FEATURE CREATION

Project: Unified Military Analytics & Comparison Dashboard

Objective: Transform raw military numbers into strategic intelligence by computing Key Performance Indicators (KPIs) and enriching with
geographic and alliance metadata.

SECTION 1: IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


SECTION 2: LOCATE AND LOAD CLEANED DATASET

Load the cleaned dataset from Milestone 01

In [ ]:
print("STEP 1: LOCATING CLEANED DATASET")
print("=" * 60)

# Search for the file
cleaned_file = None
search_paths = [
    '/content/milestone_01_data_collection/data/military_cleaned.csv',
    '/content/data/military_cleaned.csv',
    '/content/military_cleaned.csv',
    '/content/military_clean_data.csv',
    '/content/military_cleaned_data.csv',
    '/military_cleaned_data.csv', # Added this path for selected file
    '/content/drive/MyDrive/military_cleaned.csv'
]

print("🔍 Searching for military_cleaned.csv (and military_clean_data.csv, military_cleaned_data.csv)...")

# Try predefined paths
for path in search_paths:
    if os.path.exists(path):
        cleaned_file = path
        print(f"✅ Found at: {path}")
        break

# If not found, search entire /content
if cleaned_file is None:
    print("   Not in predefined paths. Searching entire /content...")
    for root, dirs, files in os.walk('/content'):
        if 'military_cleaned.csv' in files:
            cleaned_file = os.path.join(root, 'military_cleaned.csv')
            print(f"✅ Found at: {cleaned_file}")
            break
        if 'military_clean_data.csv' in files:
            cleaned_file = os.path.join(root, 'military_clean_data.csv')
            print(f"✅ Found at: {cleaned_file}")
            break
        if 'military_cleaned_data.csv' in files:
            cleaned_file = os.path.join(root, 'military_cleaned_data.csv')
            print(f"✅ Found at: {cleaned_file}")
            break

# If still not found, try to load from Google Drive
if cleaned_file is None:
    print("   Not found in /content. Checking Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path_cleaned = '/content/drive/MyDrive/military_cleaned.csv'
        drive_path_clean_data = '/content/drive/MyDrive/military_clean_data.csv'
        drive_path_cleaned_data = '/content/drive/MyDrive/military_cleaned_data.csv'
        if os.path.exists(drive_path_cleaned):
            cleaned_file = drive_path_cleaned
            print(f"✅ Found in Google Drive: {drive_path_cleaned}")
        elif os.path.exists(drive_path_clean_data):
            cleaned_file = drive_path_clean_data
            print(f"✅ Found in Google Drive: {drive_path_clean_data}")
        elif os.path.exists(drive_path_cleaned_data):
            cleaned_file = drive_path_cleaned_data
            print(f"✅ Found in Google Drive: {drive_path_cleaned_data}")
    except:
        pass

# If still not found, give instructions
if cleaned_file is None:
    print("\n" + "=" * 60)
    print("❌ ERROR: military_cleaned.csv/military_clean_data.csv/military_cleaned_data.csv NOT FOUND!")
    print("=" * 60)
    print("\nPlease upload the file manually:")
    print("1. Click the folder icon on the left sidebar")
    print("2. Click 'Upload' button")
    print("3. Select 'military_cleaned.csv', 'military_clean_data.csv' or 'military_cleaned_data.csv' from your computer")
    print("4. Then run this cell again")
    print("\nOR run Milestone 1 first to generate the file.")
    raise FileNotFoundError("military_cleaned.csv/military_clean_data.csv/military_cleaned_data.csv not found. Please upload or run Milestone 1 first.")

# Load the dataset
print("\n📂 Loading dataset...")
df = pd.read_csv(cleaned_file)
print(f"✅ Dataset loaded successfully!")
print(f"   Shape: {df.shape}")
print(f"   Countries: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"\n📋 First 5 columns: {list(df.columns[:5])}...")

# Standardize column names (ensure consistency)
df.columns = df.columns.str.strip().str.lower()

STEP 1: LOCATING CLEANED DATASET
🔍 Searching for military_cleaned.csv (and military_clean_data.csv, military_cleaned_data.csv)...
✅ Found at: /military_cleaned_data.csv

📂 Loading dataset...
✅ Dataset loaded successfully!
   Shape: (145, 34)
   Countries: 145
   Columns: 34

📋 First 5 columns: ['country', 'total_population', 'gdp', 'defense_budget', 'total_manpower']...


SECTION 3: ENSURE POWER_INDEX IS NUMERIC

In [ ]:
print("\n" + "=" * 60)
print("STEP 2: ENSURING POWER_INDEX IS NUMERIC")
print("=" * 60)

print(f"Before conversion: power_index dtype = {df['power_index'].dtype}")

# Convert power_index to numeric
df['power_index'] = pd.to_numeric(df['power_index'], errors='coerce')

# Check for any NaN values
if df['power_index'].isnull().any():
    print("⚠️  Warning: Some power_index values could not be converted:")
    print(df[df['power_index'].isnull()][['country', 'power_index']].head())
    # Fill NaN with a high number (so they rank at bottom)
    df['power_index'] = df['power_index'].fillna(999)
else:
    print("✅ All power_index values converted successfully")

print(f"After conversion: power_index dtype = {df['power_index'].dtype}")


STEP 2: ENSURING POWER_INDEX IS NUMERIC
Before conversion: power_index dtype = float64
✅ All power_index values converted successfully
After conversion: power_index dtype = float64


SECTION 4: METADATA ENRICHMENT - NATO ALLIANCE FLAG

In [ ]:
print("\n" + "=" * 60)
print("STEP 3: ADDING METADATA - NATO ALLIANCE FLAG")
print("=" * 60)

# Complete list of NATO member countries (lowercase for matching)
nato_list = [
    "albania", "belgium", "bulgaria", "canada", "croatia", "czech republic",
    "denmark", "estonia", "finland", "france", "germany", "greece",
    "hungary", "iceland", "italy", "latvia", "lithuania", "luxembourg",
    "montenegro", "netherlands", "north macedonia", "norway", "poland",
    "portugal", "romania", "slovakia", "slovenia", "spain", "sweden",
    "turkey", "united kingdom", "united states of america"
]

# Create alliance_flag column
df['alliance_flag'] = (
    df['country']
    .str.lower()
    .str.strip()
    .apply(lambda x: 'NATO' if x in nato_list else 'Non-Aligned')
)

nato_count = df['alliance_flag'].value_counts().get('NATO', 0)
non_nato_count = df['alliance_flag'].value_counts().get('Non-Aligned', 0)

print(f"✅ NATO members flagged: {nato_count} countries")
print(f"   Non-Aligned: {non_nato_count} countries")


STEP 3: ADDING METADATA - NATO ALLIANCE FLAG
✅ NATO members flagged: 32 countries
   Non-Aligned: 113 countries


SECTION 5: METADATA ENRICHMENT - GEOGRAPHIC CLASSIFICATION

In [ ]:
print("\n" + "=" * 60)
print("STEP 4: ADDING METADATA - GEOGRAPHIC CLASSIFICATION")
print("=" * 60)

def get_continent(country):
    """
    Map country to continent based on geographical location.
    """
    c = str(country).lower().strip()

    # North America
    if any(x in c for x in ['united states', 'canada', 'mexico', 'cuba', 'panama',
                             'guatemala', 'belize', 'dominican republic', 'el salvador',
                             'honduras', 'nicaragua', 'costa rica', 'jamaica', 'haiti']):
        return 'North America'

    # South America
    if any(x in c for x in ['brazil', 'argentina', 'chile', 'colombia', 'peru',
                             'ecuador', 'venezuela', 'uruguay', 'paraguay',
                             'bolivia', 'suriname', 'guyana']):
        return 'South America'

    # Asia
    if any(x in c for x in ['india', 'china', 'japan', 'south korea', 'north korea',
                             'pakistan', 'indonesia', 'vietnam', 'thailand', 'philippines',
                             'singapore', 'malaysia', 'cambodia', 'laos', 'myanmar',
                             'kazakhstan', 'uzbekistan', 'turkmenistan', 'kyrgyzstan', 'tajikistan',
                             'israel', 'saudi arabia', 'turkey', 'iran', 'iraq', 'uae', 'qatar',
                             'jordan', 'kuwait', 'lebanon', 'oman', 'syria', 'yemen', 'bahrain',
                             'afghanistan', 'bangladesh', 'sri lanka', 'nepal', 'bhutan', 'taiwan', 'mongolia']):
        return 'Asia'

    # Europe
    if any(x in c for x in ['russia', 'ukraine', 'poland', 'romania', 'belarus',
                             'czech republic', 'hungary', 'bulgaria', 'moldova', 'slovakia',
                             'france', 'germany', 'belgium', 'netherlands', 'switzerland',
                             'austria', 'luxembourg', 'united kingdom', 'sweden', 'norway',
                             'denmark', 'finland', 'ireland', 'iceland', 'estonia', 'latvia',
                             'lithuania', 'italy', 'spain', 'greece', 'portugal', 'serbia',
                             'croatia', 'slovenia', 'bosnia', 'montenegro', 'north macedonia',
                             'albania', 'kosovo', 'cyprus', 'malta']):
        return 'Europe'

    # Africa
    if any(x in c for x in ['egypt', 'algeria', 'morocco', 'libya', 'tunisia', 'sudan',
                             'south sudan', 'nigeria', 'south africa', 'kenya', 'ethiopia',
                             'angola', 'benin', 'botswana', 'burundi', 'cameroon', 'chad',
                             'congo', 'ivory coast', 'dr congo', 'eritrea', 'gabon', 'ghana',
                             'guinea', 'madagascar', 'malawi', 'mali', 'mauritania', 'mozambique',
                             'namibia', 'niger', 'rwanda', 'senegal', 'sierra leone', 'somalia',
                             'tanzania', 'uganda', 'zambia', 'zimbabwe']):
        return 'Africa'

    # Oceania
    if any(x in c for x in ['australia', 'new zealand']):
        return 'Oceania'

    return 'Other'

def get_region(country):
    """
    Map country to region based on geographical location.
    """
    c = str(country).lower().strip()

    # North America regions
    if c in ['united states', 'canada']:
        return 'Northern America'
    if any(x in c for x in ['mexico', 'cuba', 'panama', 'guatemala', 'belize', 'dominican republic',
                             'el salvador', 'honduras', 'nicaragua', 'costa rica', 'jamaica', 'haiti']):
        return 'Central America & Caribbean'

    # South America
    if any(x in c for x in ['brazil', 'argentina', 'chile', 'colombia', 'peru', 'ecuador',
                             'venezuela', 'uruguay', 'paraguay', 'bolivia', 'suriname', 'guyana']):
        return 'South America'

    # Asia regions
    if any(x in c for x in ['india', 'pakistan', 'afghanistan', 'bangladesh', 'sri lanka', 'nepal', 'bhutan']):
        return 'Southern Asia'
    if any(x in c for x in ['china', 'japan', 'south korea', 'north korea', 'taiwan', 'mongolia']):
        return 'Eastern Asia'
    if any(x in c for x in ['vietnam', 'thailand', 'indonesia', 'philippines', 'singapore',
                             'malaysia', 'cambodia', 'laos', 'myanmar']):
        return 'South-Eastern Asia'
    if any(x in c for x in ['kazakhstan', 'uzbekistan', 'turkmenistan', 'kyrgyzstan', 'tajikistan']):
        return 'Central Asia'
    if any(x in c for x in ['israel', 'saudi arabia', 'turkey', 'iran', 'iraq', 'uae', 'qatar',
                             'jordan', 'kuwait', 'lebanon', 'oman', 'syria', 'yemen', 'bahrain']):
        return 'Western Asia'

    # Europe regions
    if any(x in c for x in ['russia', 'ukraine', 'poland', 'romania', 'belarus', 'czech republic',
                             'hungary', 'bulgaria', 'moldova', 'slovakia']):
        return 'Eastern Europe'
    if any(x in c for x in ['france', 'germany', 'belgium', 'netherlands', 'switzerland',
                             'austria', 'luxembourg']):
        return 'Western Europe'
    if any(x in c for x in ['united kingdom', 'sweden', 'norway', 'denmark', 'finland', 'ireland',
                             'iceland', 'estonia', 'latvia', 'lithuania']):
        return 'Northern Europe'
    if any(x in c for x in ['italy', 'spain', 'greece', 'portugal', 'serbia', 'croatia', 'slovenia',
                             'bosnia', 'montenegro', 'north macedonia', 'albania', 'kosovo', 'cyprus', 'malta']):
        return 'Southern Europe'

    # Africa regions
    if any(x in c for x in ['egypt', 'algeria', 'morocco', 'libya', 'tunisia', 'sudan', 'south sudan']):
        return 'Northern Africa'
    if any(x in c for x in ['nigeria', 'south africa', 'kenya', 'ethiopia', 'angola', 'benin', 'botswana',
                             'burundi', 'cameroon', 'chad', 'congo', 'ivory coast', 'dr congo', 'eritrea',
                             'gabon', 'ghana', 'guinea', 'madagascar', 'malawi', 'mali', 'mauritania',
                             'mozambique', 'namibia', 'niger', 'rwanda', 'senegal', 'sierra leone',
                             'somalia', 'tanzania', 'uganda', 'zambia', 'zimbabwe']):
        return 'Sub-Saharan Africa'

    # Oceania
    if any(x in c for x in ['australia', 'new zealand']):
        return 'Australia & New Zealand'

    return 'Other'

# Apply geographic mapping
df['continent'] = df['country'].apply(get_continent)
df['region'] = df['country'].apply(get_region)


STEP 4: ADDING METADATA - GEOGRAPHIC CLASSIFICATION


SECTION 6: ENSURE NUMERIC TYPES FOR CALCULATIONS

In [ ]:
print("\n" + "=" * 60)
print("STEP 5: ENSURING NUMERIC TYPES")
print("=" * 60)

numeric_cols = ['total_aircraft', 'tanks', 'naval_assets', 'total_manpower',
                'defense_budget', 'gdp']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print(f"✅ Converted {len(numeric_cols)} columns to numeric")


STEP 5: ENSURING NUMERIC TYPES
✅ Converted 6 columns to numeric


SECTION 7: KPI CALCULATIONS

In [ ]:
print("\n" + "=" * 60)
print("STEP 6: CALCULATING KPIs")
print("=" * 60)

# KPI 1: Total Military Hardware
df['total_military_hardware'] = (
    df.get('total_aircraft', 0) +
    df.get('tanks', 0) +
    df.get('naval_assets', 0)
)

# KPI 2: Assets per Capita (Hardware per soldier)
# Use safe division (replace 0 with 1 to avoid division by zero)
df['assets_per_capita'] = (
    df['total_military_hardware'] /
    df.get('total_manpower', 1).replace(0, 1)
).round(8)

# KPI 3: Budget-to-GDP Ratio
# Use safe division (replace 0 with 1 to avoid division by zero)
df['budget_to_gdp_ratio'] = (
    df.get('defense_budget', 0) /
    df.get('gdp', 1).replace(0, 1)
).round(8)

# KPI 4: Power Index Rank (lower is better)
df['power_rank'] = df['power_index'].rank(ascending=True, method='min').astype(int)

# KPI 5: Power Index Rank Gap (distance from #1)
top_rank = df['power_rank'].min()
df['power_index_rank_gap'] = df['power_rank'] - top_rank

print(f"✅ KPIs successfully calculated:")
print(f"   - total_military_hardware: {df['total_military_hardware'].mean():.0f} avg")
print(f"   - assets_per_capita: {df['assets_per_capita'].mean():.8f} avg")
print(f"   - budget_to_gdp_ratio: {df['budget_to_gdp_ratio'].mean():.8f} avg")
print(f"   - power_rank: range {df['power_rank'].min()} to {df['power_rank'].max()}")
print(f"   - power_index_rank_gap: {df['power_index_rank_gap'].max()} max gap")


STEP 6: CALCULATING KPIs
✅ KPIs successfully calculated:
   - total_military_hardware: 431 avg
   - assets_per_capita: 0.00003631 avg
   - budget_to_gdp_ratio: 0.01946694 avg
   - power_rank: range 1 to 145
   - power_index_rank_gap: 144 max gap


SECTION 8: VALIDATE KPI VALUES

In [ ]:
print("\n" + "=" * 60)
print("STEP 7: VALIDATING KPI VALUES")
print("=" * 60)

# Check for infinite values
if np.isinf(df['budget_to_gdp_ratio']).any():
    print("⚠️  WARNING: Infinite values found in Budget-to-GDP. Replacing with 0.")
    df.replace([np.inf, -np.inf], 0, inplace=True)
else:
    print("✅ No infinite values found")

# Check for NaN values
if df[['assets_per_capita', 'budget_to_gdp_ratio', 'power_rank']].isnull().any().any():
    print("⚠️  WARNING: NaN values found in KPIs")
else:
    print("✅ No NaN values in KPIs")

# Validate with sample country (India)
india = df[df['country'].str.contains('India', case=False, na=False)]
if not india.empty:
    print("\n📊 Validation Sample (India):")
    print(f"   Country: {india['country'].values[0]}")
    print(f"   Total Manpower: {india['total_manpower'].values[0]:,.0f}")
    print(f"   Total Military Hardware: {india['total_military_hardware'].values[0]:,.0f}")
    print(f"   Assets per Capita: {india['assets_per_capita'].values[0]:.8f}")
    print(f"   Defense Budget: ${india['defense_budget'].values[0]:,.0f}")
    print(f"   GDP: ${india['gdp'].values[0]:,.0f}")
    print(f"   Budget-to-GDP Ratio: {india['budget_to_gdp_ratio'].values[0]:.6f}")
    print(f"   Power Index: {india['power_index'].values[0]}")
    print(f"   Power Rank: {india['power_rank'].values[0]}")
    print(f"   Power Rank Gap: {india['power_index_rank_gap'].values[0]}")
    print(f"   Continent: {india['continent'].values[0]}")
    print(f"   NATO Member: {'Yes' if india['alliance_flag'].values[0] == 'NATO' else 'No'}")


STEP 7: VALIDATING KPI VALUES
✅ No infinite values found
✅ No NaN values in KPIs

📊 Validation Sample (India):
   Country: India
   Total Manpower: 662,290,299
   Total Military Hardware: 2,526
   Assets per Capita: 0.00000381
   Defense Budget: $109,000,000,000
   GDP: $14,244,000,000,000
   Budget-to-GDP Ratio: 0.007652
   Power Index: 0.1346
   Power Rank: 4
   Power Rank Gap: 3
   Continent: Asia
   NATO Member: No


SECTION 9: EXPORT FINAL DATASET

In [ ]:
print("\n" + "=" * 60)
print("STEP 8: EXPORTING FINAL DATASET")
print("=" * 60)

# Create data folder if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')
    print("✅ Created 'data' folder")

# Save final dataset
OUTPUT_FILE = "data/military_final.csv"
df.to_csv(OUTPUT_FILE, index=False)

print(f"✅ Final dataset saved: {OUTPUT_FILE}")
print(f"📊 Final shape: {df.shape}")
print(f"📁 File size: {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")


STEP 8: EXPORTING FINAL DATASET
✅ Final dataset saved: data/military_final.csv
📊 Final shape: (145, 42)
📁 File size: 31.5 KB


SECTION 10: SUMMARY STATISTICS

In [ ]:
print("\n" + "=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)

print(f"\n📊 Dataset Overview:")
print(f"   Countries: {len(df)}")
print(f"   Total Columns: {len(df.columns)}")
print(f"   New Columns Added: {len([c for c in df.columns if c not in numeric_cols + ['country', 'power_index']])}")

print(f"\n📈 Top 10 Countries by Power Rank:")
top_10 = df.nsmallest(10, 'power_rank')[['country', 'power_rank', 'power_index', 'assets_per_capita', 'budget_to_gdp_ratio']]
print(top_10.to_string(index=False))

print(f"\n🛡️ NATO vs Non-NATO Summary:")
nato_summary = df.groupby('alliance_flag').agg({
    'defense_budget': 'sum',
    'total_manpower': 'sum',
    'total_military_hardware': 'sum',
    'power_rank': 'mean'
}).round(2)
print(nato_summary.to_string())

print(f"\n🌍 Continent Summary:")
continent_summary = df.groupby('continent').agg({
    'defense_budget': 'sum',
    'total_manpower': 'sum',
    'country': 'count'
}).round(2)
continent_summary.rename(columns={'country': 'countries'}, inplace=True)
print(continent_summary.to_string())

print("\n" + "=" * 60)
print("MILESTONE 02 COMPLETE!")
print("=" * 60)
print("\n📌 Next: Milestone 03 - Dashboard Development")
print("   Input: military_final.csv")
print("   Output: Interactive dashboards with 4 modules")


FINAL DATASET SUMMARY

📊 Dataset Overview:
   Countries: 145
   Total Columns: 42
   New Columns Added: 34

📈 Top 10 Countries by Power Rank:
                 country  power_rank  power_index  assets_per_capita  budget_to_gdp_ratio
United States Of America           1       0.0741           0.000090             0.032384
                  Russia           2       0.0791           0.000072             0.034922
                   China           3       0.0919           0.000006             0.009018
                   India           4       0.1346           0.000004             0.007652
             South Korea           5       0.1642           0.000067             0.017184
                  France           6       0.1798           0.000038             0.018015
                   Japan           7       0.1876           0.000030             0.010149
          United Kingdom           8       0.1881           0.000024             0.024347
                  Turkey           9       0.19